In [1]:
%cd ..

/Users/po3na4skld/PycharmProjects/Personal/SynSteg


/Users/po3na4skld/PycharmProjects/Binerals/bin_datasets_reading/venv/lib/python3.11/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import json
import copy
from functools import reduce

from glob import glob
import numpy as np

from models.report import ReportModel
from steganography.core import decode_message
from steganography.secret_key import binarize_synonyms_partially
from steganography import helper

PROMPT_PRICE = 5.0e-6
COMPLETION_PRICE = 15.0e-6

In [14]:
JSONS_PATH = 'reports/gpt_omni_reports/*.json'
jsons = glob(JSONS_PATH)

In [30]:
def read_report(path_to_report: str) -> ReportModel:
    with open(path_to_report) as f:
        report = ReportModel(**json.load(f))
        if not report.decoded_message:
            report.decoded_message = decode_message(report.encoded_message, report.secret_key, clean_output=False)
        return report

    
def add_additional_bits_to_reports(reports: list[ReportModel], bits_per_word: int, additional_bits: int) -> list[ReportModel]:
    new_reports = []
    chunk_size = bits_per_word + additional_bits
    for report in reports:
        new_report = copy.deepcopy(report) 
        binary_message_chunks = list(helper.divide_chunks(report.message, chunk_size))
        new_secret_key, is_filled = [], False
        for idx, secret_key_token in enumerate(report.secret_key):
            if is_filled:
                break
            for key, value in secret_key_token.items():
                try:
                    binary_message_chunk = binary_message_chunks[idx]
                    partially_binarized = binarize_synonyms_partially(list(set(value.values())), binary_message_chunk)
                    new_secret_key.append({key: partially_binarized})
                except IndexError:
                    is_filled = True
    
        new_report.secret_key = new_secret_key
        new_reports.append(new_report)
    return new_reports
    
    
def get_secret_key_strength(reports: list[ReportModel], message_length: int, bits_per_word: int) -> float:
    filtered_reports = [report for report in reports if len(report.message) == message_length]
    strengths = []
    for report in filtered_reports:
        n_of_elements = []
        for synonym_map in report.secret_key:
            for key, value in synonym_map.items():
                n_of_elements.append(len(value))

        strengths.append(reduce(lambda x, y: x * y, n_of_elements))
    
    print(f"Mean secret key strength for {message_length} and {bits_per_word} bits per word: {np.mean(strengths)}")
    
    
def get_throughput(report: ReportModel) -> float:
    """Function that calculates throughtput: volume of secret message / volume of the container"""
    
    n_clean_words = len(report.container.split()) - len(report.secret_key)
    return len(report.message) / (len(report.container.split()) - n_clean_words)


def get_mean_and_std(sequence: list[float], title: str) -> None:
    print(title)
    print('Mean:', round(np.mean(sequence), 4), 'STD:', round(np.std(sequence), 4))


def calculate_average_er(L, m, k):
    return ((L - 1) * k) / (8 * L * m)
    
    
def get_embedding_rate(secret_key_avg_bit_size: float, filtered_reports: list[ReportModel], message_length: int):
    L = round(np.mean([len(i.split()) for report in filtered_reports for i in report.encoded_message.split('.')]), 2)
    M = round(np.mean([len(i) for report in filtered_reports for i in report.encoded_message.split()]), 2)
    embedding_rate = round(calculate_average_er(L, M, secret_key_avg_bit_size), 4)
    
    print(f'The average length of each sentence in the generated text for {message_length} bit message length: {L}')
    print(f'The average number of letters contained in each word for {message_length} bit message length: {M}')
    print(f'The embedding rate for {message_length} bit message length: {embedding_rate}')



def get_metrics(reports: list[ReportModel], message_length: int):
    
    filtered_reports = [report for report in reports if len(report.message) == message_length]
    
    throughput = [get_throughput(report) for report in filtered_reports]
    get_mean_and_std(throughput, f'Throughput {message_length} bits, bit/word')
    print('--' * 10)
    
    speed = [report.encoding_time_report['container_generation'] for report in filtered_reports]
    get_mean_and_std(speed, 'Container generation speed, seconds')
    print('--' * 10)
    
    speed = [report.encoding_time_report['secret_key_generation'] for report in filtered_reports]
    get_mean_and_std(speed, 'Secret key generation speed, seconds')
    print('--' * 10)
    
    speed = [report.decoding_time for report in filtered_reports]
    get_mean_and_std(speed, 'Decoding/Encoding time speed, seconds')
    print('--' * 10)
    
    secret_key_length = [len(report.secret_key) for report in filtered_reports]
    get_mean_and_std(secret_key_length, 'Secret key and Container length, words')
    print('--' * 10)
    
    container_completion_tokens = [
        report.encoding_usage_report["container_generation"]["completion_tokens"] 
        for report in filtered_reports
    ]
    
    container_prompt_tokens = [
        report.encoding_usage_report["container_generation"]["prompt_tokens"]
        for report in filtered_reports
    ]

    get_mean_and_std(container_completion_tokens, 'Completion tokens number for Container generation')
    print('--' * 10)


    get_mean_and_std(
        [i * COMPLETION_PRICE for i in container_completion_tokens], 
        'Completion tokens USD price for Container generation'
    )
    print('--' * 10)

    get_mean_and_std(container_prompt_tokens, 'Prompt tokens number for Container generation')
    print('--' * 10)

    get_mean_and_std(
        [i * PROMPT_PRICE for i in container_prompt_tokens], 
        'Prompt tokens USD price for Container generation'
    )
    print('--' * 10)

    secret_key_completion_tokens = [
        sum(usage["completion_tokens"] for usage in report.encoding_usage_report["secret_key_generation"]) 
        for report in filtered_reports
    ]
    
    secret_key_prompt_tokens = [
        sum(usage["prompt_tokens"] for usage in report.encoding_usage_report["secret_key_generation"]) 
        for report in filtered_reports
    ]

    get_mean_and_std(secret_key_completion_tokens, 'Completion tokens number for Secret Key generation')
    print('--' * 10)

    get_mean_and_std(
        [i * COMPLETION_PRICE for i in secret_key_completion_tokens], 
        'Completion tokens USD price for Secret Key generation'
    )
    print('--' * 10)


    get_mean_and_std(secret_key_prompt_tokens, 'Prompt tokens number for Secret Key generation')
    print('--' * 10)

    get_mean_and_std(
        [i * PROMPT_PRICE for i in secret_key_prompt_tokens], 
        'Prompt tokens USD price for Secret Key generation'
    )
    print('--' * 10)
    
    secret_key_average_word_size = []
    for report in filtered_reports:
        for secret_key_token in report.secret_key:
            for v in secret_key_token.values():
                for k in v:
                    secret_key_average_word_size.append(len(k))
                    break
                break

    get_mean_and_std(secret_key_average_word_size, 'Secret key token bits size')
    print('--' * 10)
    
    get_embedding_rate(np.mean(secret_key_average_word_size), filtered_reports, message_length)
    print('--' * 10)
    
    print('End of metrics')

In [15]:
REPORTS = [read_report(report_path) for report_path in jsons] 

## Metrics for 128 bit message length

In [16]:
get_metrics(REPORTS, 128)

Throughput 128 bits, bit/word
Mean: 4.3403 STD: 0.1946
--------------------
Container generation speed, seconds
Mean: 1.1662 STD: 1.0988
--------------------
Secret key generation speed, seconds
Mean: 30.1971 STD: 38.6952
--------------------
Decoding/Encoding time speed, seconds
Mean: 0.0009 STD: 0.0002
--------------------
Secret key and Container length, words
Mean: 29.55 STD: 1.3219
--------------------
Completion tokens number for Container generation
Mean: 35.26 STD: 2.1242
--------------------
Completion tokens USD price for Container generation
Mean: 0.0005 STD: 0.0
--------------------
Prompt tokens number for Container generation
Mean: 107.99 STD: 1.8083
--------------------
Prompt tokens USD price for Container generation
Mean: 0.0005 STD: 0.0
--------------------
Completion tokens number for Secret Key generation
Mean: 4415.72 STD: 324.7849
--------------------
Completion tokens USD price for Secret Key generation
Mean: 0.0662 STD: 0.0049
--------------------
Prompt tokens 

## Metrics for 256 bit message length

In [17]:
get_metrics(REPORTS, 256)

Throughput 256 bits, bit/word
Mean: 4.269 STD: 0.1583
--------------------
Container generation speed, seconds
Mean: 1.2261 STD: 0.3841
--------------------
Secret key generation speed, seconds
Mean: 21.6551 STD: 2.6266
--------------------
Decoding/Encoding time speed, seconds
Mean: 0.0017 STD: 0.0003
--------------------
Secret key and Container length, words
Mean: 60.05 STD: 2.2198
--------------------
Completion tokens number for Container generation
Mean: 70.98 STD: 3.3226
--------------------
Completion tokens USD price for Container generation
Mean: 0.0011 STD: 0.0
--------------------
Prompt tokens number for Container generation
Mean: 108.0 STD: 1.6062
--------------------
Prompt tokens USD price for Container generation
Mean: 0.0005 STD: 0.0
--------------------
Completion tokens number for Secret Key generation
Mean: 8974.55 STD: 510.6884
--------------------
Completion tokens USD price for Secret Key generation
Mean: 0.1346 STD: 0.0077
--------------------
Prompt tokens num

## Metrics for 512 bit message length

In [18]:
get_metrics(REPORTS, 512)

Throughput 512 bits, bit/word
Mean: 4.0997 STD: 0.0928
--------------------
Container generation speed, seconds
Mean: 1.9318 STD: 0.3944
--------------------
Secret key generation speed, seconds
Mean: 22.9657 STD: 1.9525
--------------------
Decoding/Encoding time speed, seconds
Mean: 0.0031 STD: 0.0004
--------------------
Secret key and Container length, words
Mean: 124.95 STD: 2.8439
--------------------
Completion tokens number for Container generation
Mean: 146.48 STD: 5.7558
--------------------
Completion tokens USD price for Container generation
Mean: 0.0022 STD: 0.0001
--------------------
Prompt tokens number for Container generation
Mean: 108.1 STD: 1.6462
--------------------
Prompt tokens USD price for Container generation
Mean: 0.0005 STD: 0.0
--------------------
Completion tokens number for Secret Key generation
Mean: 18762.88 STD: 757.9389
--------------------
Completion tokens USD price for Secret Key generation
Mean: 0.2814 STD: 0.0114
--------------------
Prompt tok

## Prepare reports for additional bits

In [19]:
REPORTS_WITH_ADDITIONAL_BITS = [read_report(report_path) for report_path in jsons]
BITS_PER_WORD = 5

In [20]:
REPORTS_WITH_5_ADDITIONAL_BITS = add_additional_bits_to_reports(REPORTS_WITH_ADDITIONAL_BITS, BITS_PER_WORD, 5)
REPORTS_WITH_10_ADDITIONAL_BITS = add_additional_bits_to_reports(REPORTS_WITH_ADDITIONAL_BITS, BITS_PER_WORD, 10)
REPORTS_WITH_15_ADDITIONAL_BITS = add_additional_bits_to_reports(REPORTS_WITH_ADDITIONAL_BITS, BITS_PER_WORD, 15)

## Metrics for 128 bit message length with additional bits = 5

In [22]:
get_metrics(REPORTS_WITH_5_ADDITIONAL_BITS, 128)

Throughput 128 bits, bit/word
Mean: 9.8462 STD: 0.0
--------------------
Container generation speed, seconds
Mean: 1.1662 STD: 1.0988
--------------------
Secret key generation speed, seconds
Mean: 30.1971 STD: 38.6952
--------------------
Decoding/Encoding time speed, seconds
Mean: 0.0009 STD: 0.0002
--------------------
Secret key and Container length, words
Mean: 13.0 STD: 0.0
--------------------
Completion tokens number for Container generation
Mean: 35.26 STD: 2.1242
--------------------
Completion tokens USD price for Container generation
Mean: 0.0005 STD: 0.0
--------------------
Prompt tokens number for Container generation
Mean: 107.99 STD: 1.8083
--------------------
Prompt tokens USD price for Container generation
Mean: 0.0005 STD: 0.0
--------------------
Completion tokens number for Secret Key generation
Mean: 4415.72 STD: 324.7849
--------------------
Completion tokens USD price for Secret Key generation
Mean: 0.0662 STD: 0.0049
--------------------
Prompt tokens number 

## Metrics for 256 bit message length with additional bits = 5

In [23]:
get_metrics(REPORTS_WITH_5_ADDITIONAL_BITS, 256)

Throughput 256 bits, bit/word
Mean: 9.8462 STD: 0.0
--------------------
Container generation speed, seconds
Mean: 1.2261 STD: 0.3841
--------------------
Secret key generation speed, seconds
Mean: 21.6551 STD: 2.6266
--------------------
Decoding/Encoding time speed, seconds
Mean: 0.0017 STD: 0.0003
--------------------
Secret key and Container length, words
Mean: 26.0 STD: 0.0
--------------------
Completion tokens number for Container generation
Mean: 70.98 STD: 3.3226
--------------------
Completion tokens USD price for Container generation
Mean: 0.0011 STD: 0.0
--------------------
Prompt tokens number for Container generation
Mean: 108.0 STD: 1.6062
--------------------
Prompt tokens USD price for Container generation
Mean: 0.0005 STD: 0.0
--------------------
Completion tokens number for Secret Key generation
Mean: 8974.55 STD: 510.6884
--------------------
Completion tokens USD price for Secret Key generation
Mean: 0.1346 STD: 0.0077
--------------------
Prompt tokens number fo

## Metrics for 512 bit message length with additional bits = 5

In [24]:
get_metrics(REPORTS_WITH_5_ADDITIONAL_BITS, 512)

Throughput 512 bits, bit/word
Mean: 9.8462 STD: 0.0
--------------------
Container generation speed, seconds
Mean: 1.9318 STD: 0.3944
--------------------
Secret key generation speed, seconds
Mean: 22.9657 STD: 1.9525
--------------------
Decoding/Encoding time speed, seconds
Mean: 0.0031 STD: 0.0004
--------------------
Secret key and Container length, words
Mean: 52.0 STD: 0.0
--------------------
Completion tokens number for Container generation
Mean: 146.48 STD: 5.7558
--------------------
Completion tokens USD price for Container generation
Mean: 0.0022 STD: 0.0001
--------------------
Prompt tokens number for Container generation
Mean: 108.1 STD: 1.6462
--------------------
Prompt tokens USD price for Container generation
Mean: 0.0005 STD: 0.0
--------------------
Completion tokens number for Secret Key generation
Mean: 18762.88 STD: 757.9389
--------------------
Completion tokens USD price for Secret Key generation
Mean: 0.2814 STD: 0.0114
--------------------
Prompt tokens numb

## Strength of secret key with VS without additional bits

### 128 bit message length

In [33]:
get_secret_key_strength(REPORTS, 128, 5)
get_secret_key_strength(REPORTS_WITH_5_ADDITIONAL_BITS, 128, 10)
get_secret_key_strength(REPORTS_WITH_10_ADDITIONAL_BITS, 128, 15)
get_secret_key_strength(REPORTS_WITH_15_ADDITIONAL_BITS, 128, 20)

Mean secret key strength for 128 and 5 bits per word: 5.784821880677872e+46
Mean secret key strength for 128 and 10 bits per word: 1.0670148768259135e+19
Mean secret key strength for 128 and 15 bits per word: 14369582512198.88
Mean secret key strength for 128 and 20 bits per word: 18428736308.24


### 256 bit message length

In [31]:
get_secret_key_strength(REPORTS, 256, 5)
get_secret_key_strength(REPORTS_WITH_5_ADDITIONAL_BITS, 256, 10)
get_secret_key_strength(REPORTS_WITH_10_ADDITIONAL_BITS, 256, 15)
get_secret_key_strength(REPORTS_WITH_15_ADDITIONAL_BITS, 256, 20)

Mean secret key strength for 256 and 5 bits per word: 2.2136744043544617e+94
Mean secret key strength for 256 and 10 bits per word: 9.124586316758766e+37
Mean secret key strength for 256 and 15 bits per word: 1.721970397811539e+26
Mean secret key strength for 256 and 20 bits per word: 9.112824934709622e+18


### 512 bit message length

In [32]:
get_secret_key_strength(REPORTS, 512, 5)
get_secret_key_strength(REPORTS_WITH_5_ADDITIONAL_BITS, 512, 10)
get_secret_key_strength(REPORTS_WITH_10_ADDITIONAL_BITS, 512, 15)
get_secret_key_strength(REPORTS_WITH_15_ADDITIONAL_BITS, 512, 20)

Mean secret key strength for 512 and 5 bits per word: 3.673690359840068e+192
Mean secret key strength for 512 and 10 bits per word: 7.515307608707255e+75
Mean secret key strength for 512 and 15 bits per word: 1.1827373503337231e+51
Mean secret key strength for 512 and 20 bits per word: 7.675180366832541e+37
